#### Loading the Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

#### Loading the Feature-Engineered Dataset

In [3]:
df = pd.read_csv(
    '../data/processed/feature_engineered_data.csv'
)

df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Region,TrafficType,VisitorType,Weekend,Revenue,Total_Pages_Viewed,Total_Browsing_Duration,Product_Page_Share,Avg_Duration_Per_Page,Product_Duration_Share
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,1,1,Returning_Visitor,False,False,1,0.000000,1.0,0.000000,0.0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,1,2,Returning_Visitor,False,False,2,64.000000,1.0,32.000000,1.0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,9,3,Returning_Visitor,False,False,1,0.000000,1.0,0.000000,0.0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,2,4,Returning_Visitor,False,False,2,2.666667,1.0,1.333333,1.0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,1,4,Returning_Visitor,True,False,10,627.500000,1.0,62.750000,1.0


#### Creating the Modeling Dataset

In [4]:
df_model = df.copy()

#### Separating Features and Target

In [5]:
X = df_model.drop(
    'Revenue',
    axis=1
)

y = df_model['Revenue']

#### Checking the Feature and Target Shapes

In [6]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (12205, 22)
Target shape: (12205,)


#### Creating the Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### Checking the Split Shapes

In [8]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (9764, 22)
X_test shape: (2441, 22)
y_train shape: (9764,)
y_test shape: (2441,)


#### Checking Target Distribution

In [9]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training target distribution:
Revenue
False    0.843712
True     0.156288
Name: proportion, dtype: float64

Testing target distribution:
Revenue
False    0.843507
True     0.156493
Name: proportion, dtype: float64


#### Creating the Training Dataset

In [10]:
train_data = X_train.copy()
train_data['Revenue'] = y_train.values

#### Creating the Testing Dataset

In [11]:
test_data = X_test.copy()
test_data['Revenue'] = y_test.values

#### Saving the Training Dataset

In [12]:
train_data.to_csv(
    '../data/processed/train_data.csv',
    index=False
)

#### Saving the Testing Dataset

In [13]:
test_data.to_csv(
    '../data/processed/test_data.csv',
    index=False
)

#### Checking the Saved Dataset Shapes

In [14]:
print("Training dataset shape:", train_data.shape)
print("Testing dataset shape:", test_data.shape)

Training dataset shape: (9764, 23)
Testing dataset shape: (2441, 23)


#### Identifying Numerical and Categorical Features

In [17]:
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64', 'bool']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object', 'category', 'str']
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Total_Pages_Viewed', 'Total_Browsing_Duration', 'Product_Page_Share', 'Avg_Duration_Per_Page', 'Product_Duration_Share']

Categorical features:
['Month', 'VisitorType']


OperatingSystems, Browser, Region, and TrafficType should be treated as categorical, not numerical. They are encoded category IDs, not quantities.

Also, Weekend is a binary feature, so we'll handle it separately as numeric/binary.

#### Defining Numerical and Categorical Features

In [18]:
numerical_features = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'BounceRates',
    'ExitRates',
    'PageValues',
    'SpecialDay',
    'Total_Pages_Viewed',
    'Total_Browsing_Duration',
    'Product_Page_Share',
    'Avg_Duration_Per_Page',
    'Product_Duration_Share'
]

categorical_features = [
    'Month',
    'OperatingSystems',
    'Browser',
    'Region',
    'TrafficType',
    'VisitorType'
]

binary_features = [
    'Weekend'
]

#### Checking the Feature Groups

In [19]:
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Binary features:", len(binary_features))

print(
    "\nTotal features:",
    len(numerical_features)
    + len(categorical_features)
    + len(binary_features)
)

Numerical features: 15
Categorical features: 6
Binary features: 1

Total features: 22


#### Importing Preprocessing Libraries

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Creating the Preprocessing Pipeline

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'numerical',
            StandardScaler(),
            numerical_features
        ),
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        ),
        (
            'binary',
            'passthrough',
            binary_features
        )
    ]
)

#### Importing SMOTE and Pipeline

In [23]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

#### Creating the SMOTE Configuration

In [24]:
smote = SMOTE(
    random_state=42
)

#### Baseline Model

In [25]:
from sklearn.linear_model import LogisticRegression

Creating the Logistic Regression Pipeline

In [26]:
logistic_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

Training the Logistic Regression Model

In [27]:
logistic_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[bool](2,)","[False, True]"
feature_names_in_,"ndarray[object](22,)","['Administrative','Administrative_Duration','Informational',..., 'Product_Page_Share','Avg_Duration_Per_Page','Product_Duration_Share']"
n_features_in_,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


Creating the Baseline Prediction

In [28]:
baseline_prediction = y_train.mode()[0]

y_test_baseline = np.full(
    len(y_test),
    baseline_prediction
)

Checking the Baseline Prediction

In [29]:
print("Baseline prediction:", baseline_prediction)
print("Baseline predictions:", len(y_test_baseline))

Baseline prediction: False
Baseline predictions: 2441


Baseline is established: it predicts False for every session.

Now we move to the first real model and, importantly, start our MLflow experiment tracking.

#### Setting the MLflow Experiment

In [31]:
import mlflow
import mlflow.sklearn

Setting the MLflow Experiment

In [32]:
mlflow.set_experiment(
    "E-commerce Conversion Dynamics"
)

2026/09/13 22:17:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/13 22:17:21 INFO mlflow.store.db.utils: Updating database tables
2026/09/13 22:17:26 INFO mlflow.tracking.fluent: Experiment with name 'E-commerce Conversion Dynamics' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///d:/Projects/ML and Data Science/E-Commerce Conversion '
 'Dynamics/notebooks/mlruns/1'), creation_time=1789318046291, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789318046291, lifecycle_stage='active', name='E-commerce Conversion Dynamics', tags={}, trace_location=None, workspace='default'>

Configuring MLflow Tracking

In [36]:
mlflow.set_tracking_uri(
    "sqlite:///../mlflow.db"
)

mlflow.set_experiment(
    "E-commerce Conversion Dynamics"
)

2026/09/13 22:32:34 INFO mlflow.tracking.fluent: Experiment with name 'E-commerce Conversion Dynamics' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///d:/Projects/ML and Data Science/E-Commerce Conversion '
 'Dynamics/notebooks/mlruns/1'), creation_time=1789318954296, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789318954296, lifecycle_stage='active', name='E-commerce Conversion Dynamics', tags={}, trace_location=None, workspace='default'>

Checking the MLflow Tracking Location

In [37]:
print(mlflow.get_tracking_uri())

sqlite:///../mlflow.db


Checking the MLflow Experiment

In [38]:
experiment = mlflow.get_experiment_by_name(
    "E-commerce Conversion Dynamics"
)

print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)

Experiment ID: 1
Experiment Name: E-commerce Conversion Dynamics


Starting the Logistic Regression MLflow Run

In [39]:
with mlflow.start_run(run_name="Logistic Regression"):

    logistic_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

Saving the Logistic Regression Pipeline

In [40]:
from joblib import dump

dump(
    logistic_pipeline,
    '../models/logistic_regression_pipeline.joblib'
)

['../models/logistic_regression_pipeline.joblib']

#### Ridge Classifier

In [41]:
from sklearn.linear_model import RidgeClassifier

In [42]:
## Creating the Ridge Classifier Pipeline

ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            RidgeClassifier(
                random_state=42
            )
        )
    ]
)

In [43]:
## Starting the Ridge Classifier MLflow Run

with mlflow.start_run(run_name="Ridge Classifier"):

    ridge_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Ridge Classifier"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

In [44]:
## Saving the Ridge Classifier Pipeline

dump(
    ridge_pipeline,
    '../models/ridge_classifier_pipeline.joblib'
)

['../models/ridge_classifier_pipeline.joblib']